# 02 - Detection de visages

Objectif : charger InsightFace, capturer une image webcam, detecter les visages et afficher les rectangles de detection.

Important : InsightFace utilise un modele pre-entraine. On ne l'entraine pas a partir de zero.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config.settings import get_settings
from src.camera.webcam import open_camera, read_frame, release_camera
from src.face.detector import InsightFaceDetector

settings = get_settings()
settings.model_name

## Capturer une image

Placez un seul visage devant la camera pour ce premier test. Les tests avec plusieurs visages viendront plus tard.

In [ ]:
capture = open_camera(settings.camera_index)
try:
    frame = read_frame(capture)
finally:
    release_camera(capture)

frame.shape

## Charger InsightFace

La premiere execution peut prendre du temps, car le modele peut etre telecharge dans le cache local d'InsightFace.

In [ ]:
detector = InsightFaceDetector(model_name=settings.model_name)
detections = detector.detect(frame)
print(f"Nombre de visages detectes : {len(detections)}")

for index, detection in enumerate(detections, start=1):
    print(index, detection.bbox, detection.confidence)

## Afficher les bounding boxes

Chaque rectangle correspond a un visage detecte. Le score affiche est la confiance de detection, pas encore l'identite de l'etudiant.

In [ ]:
import cv2
import matplotlib.pyplot as plt

preview = frame.copy()
for detection in detections:
    x1, y1, x2, y2 = detection.bbox
    cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 255, 0), 2)
    cv2.putText(
        preview,
        f"{detection.confidence:.2f}",
        (x1, max(y1 - 8, 20)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 0),
        2,
    )

rgb_preview = cv2.cvtColor(preview, cv2.COLOR_BGR2RGB)
plt.figure(figsize=(8, 5))
plt.imshow(rgb_preview)
plt.axis("off")
plt.show()

## Ce qu'il faut observer

- Si aucun visage n'est detecte, ameliorer l'eclairage et rapprocher le visage.
- Si plusieurs visages sont detectes, c'est normal : l'etape d'enregistrement exigera exactement un visage.
- L'identification viendra dans le notebook `03_face_embedding.ipynb`.